# 02 — Model Evaluation

Evaluate the trained risk model on a **fresh** synthetic draw (lots the
model never saw) and inspect what drives its predictions.

In [ ]:
# Make the `lotiq` package importable whether Jupyter launched from the repo
# root or the notebooks/ folder.
import sys, pathlib
try:
    import lotiq  # noqa: F401
except ModuleNotFoundError:
    for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_p / "src" / "lotiq").exists():
            sys.path.insert(0, str(_p / "src")); break
    import lotiq  # noqa: F401
print("lotiq", lotiq.__version__)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from lotiq.config import MODEL_FEATURES, TARGET, METRICS_PATH, band_for_score
from lotiq.data.generate import generate_dataset
from lotiq.data.preprocessing import build_feature_frame
from lotiq.models.predict import load_bundle, score_record

print(json.loads(open(METRICS_PATH).read()))
bundle = load_bundle()
print('backend:', bundle['backend'])

## Held-out evaluation

Generate fresh lots, predict, and score against the true synthetic risk.

In [ ]:
fresh = generate_dataset(n=600, seed=123)
X = build_feature_frame(fresh)
y = fresh[TARGET].to_numpy()
pred = np.clip(bundle['model'].predict(X), 0, 100)

mae = mean_absolute_error(y, pred)
rmse = np.sqrt(mean_squared_error(y, pred))
r2 = r2_score(y, pred)
band_acc = np.mean([band_for_score(a).name == band_for_score(p).name
                    for a, p in zip(y, pred)])
print(f'MAE {mae:.2f} | RMSE {rmse:.2f} | R2 {r2:.3f} | band acc {band_acc*100:.1f}%')

In [ ]:
plt.figure(figsize=(5.5, 5.5))
plt.scatter(y, pred, s=10, alpha=0.35, color='#3b6fb0')
plt.plot([0, 100], [0, 100], 'k--', lw=1)
plt.xlabel('Actual risk'); plt.ylabel('Predicted risk')
plt.title('Predicted vs actual (held-out)')
plt.tight_layout(); plt.show()

## What drives predictions?

If XGBoost + SHAP are installed, we show a SHAP summary. Otherwise we
estimate global importance from the ablation explainer across a sample.

In [ ]:
from lotiq.explain.attribution import explain_instance

used_shap = False
if bundle['backend'] == 'xgboost':
    try:
        import shap
        expl = shap.TreeExplainer(bundle['model'])
        sv = expl.shap_values(X.sample(min(300, len(X)), random_state=0)[MODEL_FEATURES])
        shap.summary_plot(sv, X[MODEL_FEATURES].iloc[:sv.shape[0]], show=True)
        used_shap = True
    except Exception as e:
        print('SHAP unavailable, using ablation:', e)

if not used_shap:
    sample = X.sample(min(150, len(X)), random_state=0).reset_index(drop=True)
    mags = np.zeros(len(MODEL_FEATURES))
    for i in range(len(sample)):
        contribs = explain_instance(bundle, sample.iloc[[i]])
        for c in contribs:
            mags[MODEL_FEATURES.index(c.feature)] += abs(c.contribution)
    mags /= len(sample)
    order = np.argsort(mags)
    plt.figure(figsize=(8, 4))
    plt.barh([MODEL_FEATURES[i] for i in order], mags[order], color='#3b6fb0')
    plt.title('Mean |contribution| (ablation)'); plt.xlabel('risk points')
    plt.tight_layout(); plt.show()

## A single explained lot

End-to-end: the score, the drivers, and the plain-language explanation the
API and dashboard return.

In [ ]:
worst = fresh.sort_values(TARGET, ascending=False).iloc[0]
record = {k: worst[k] for k in ['lot_id','commodity','temperature','humidity',
          'moisture_content','co2_level','oleoresin_content','colour_score',
          'storage_days','initial_quality_grade']}
out = score_record(record)
print(out['lot_id'], '->', out['risk_score'], out['risk_level'])
print('Factors:', out['risk_factors'])
print('Action :', out['recommendation'])
print('Why    :', out['explanation'])